In [1]:
import os
import glob
import numpy as np
from pathlib import Path
import pyarrow.parquet as pq
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers import Tokenizer, AddedToken
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from huggingface_hub import hf_hub_download

In [4]:
FILES = sorted(glob.glob(os.path.expanduser(
    "~/.cache/huggingface/hub/datasets--HuggingFaceFW--fineweb-edu"
    "/snapshots/*/sample/10BT/*.parquet")))

In [8]:
tokenizer_path = Path("./tokenizer/tokenizer.json")
tokenizer = Tokenizer.from_file(str(tokenizer_path))

In [8]:
def iter_documents_parquet(files, batch_size=1000):
    for f in files:
        for batch in pq.ParquetFile(f).iter_batches(batch_size=batch_size, columns=["text"]):
            for t in batch.column("text").to_pylist():
                t = t.strip()
                if t:
                    yield t

In [9]:
def iter_batches(FILES, batch_size=1000):
    batch = []
    for doc in iter_documents_parquet(FILES):
        batch.append(doc)
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

In [32]:
sep="<|endoftext|>"
EOS = tokenizer.token_to_id(sep)   # 1


def write_bin(path_in, path_out, batch_size=10_000):
    n_docs = n_tokens = 0
    with open(path_out, "wb") as f:
        for batch in iter_batches(path_in, batch_size):
            ids = []
            for enc in tokenizer.encode_batch_fast(batch):
                ids.extend(enc.ids)
                ids.append(EOS)
            np.asarray(ids, dtype=np.uint16).tofile(f)#tofile on an open handle appends,
            n_docs += len(batch)
            n_tokens += len(ids)                
                
    return n_docs, n_tokens

In [33]:
%%time

path_out = Path("./data/tokenized_fineweb.bin")
write_bin(FILES, path_out, batch_size=10_000)

CPU times: user 8h 49min 13s, sys: 1min 22s, total: 8h 50min 36s
Wall time: 27min 52s


(5131000, 5285351819)

In [2]:
path_out = Path("./data/tokenized_fineweb.bin")
data = np.memmap(path_out, dtype=np.uint16, mode="r")

In [3]:
type(data), data.shape

(numpy.memmap, (5285351819,))

In [ ]:
#last 20M tokens for validation
end = 20_000_000

In [11]:
data[:-end].shape, (data[:-end]==EOS).sum()

((5265351819,), np.int64(5111689))

In [12]:
data[-end:].shape, (data[-end:]==EOS).sum()

((20000000,), np.int64(19312))